In [ ]:
# --- Section 1: Header ---
# Package definition + metadata + zero-cost policy statement (M2-P0-13).
import json, os, sys

PACKAGE = json.loads(r'''
__GHARIBO_PACKAGE_JSON__
''')

assert PACKAGE.get('preview') is None, 'PREVIEW packages cannot execute'

print('=' * 72)
print('GHARIBO AI LAB - Training Package')
print('=' * 72)
print('experiment_id  :', PACKAGE['experiment_id'])
print('package_id     :', PACKAGE['package_id'])
print('schema_version :', PACKAGE['schema_version'])
print('base_model     :', PACKAGE['base_model'], '@', PACKAGE['base_model_revision'])
print('loader_model   :', PACKAGE['loader_model_id'])
print('dataset_hash   :', PACKAGE['dataset']['dataset_hash'])
print('engine         :', PACKAGE['engine']['engine'], PACKAGE['engine']['engine_version'])
print('dtype/seq_len  :', PACKAGE['dtype'], '/', PACKAGE['sequence_length'])
print('')
print('ZERO-COST POLICY: this run executes on the free Kaggle tier only.')
print('No paid training provider and no paid storage are used anywhere.')

# The package declares evaluation intent only - no metrics are ever claimed here.
assert PACKAGE['evaluation_config']['executed'] is False
assert PACKAGE['evaluation_config']['status'] == 'NOT_RUN'
print('evaluation_config: NOT_RUN (declared intent only)')


In [ ]:
# --- Section 2: Hardware detect ---
# Print GPU name, compute capability, VRAM and selected dtype (M2-P0-05).
import torch

print('torch:', torch.__version__)
assert torch.cuda.is_available(), 'No CUDA GPU detected - enable a Kaggle GPU accelerator.'
gpu_name = torch.cuda.get_device_name(0)
cap_major, cap_minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
compute_capability = 'sm_%d%d' % (cap_major, cap_minor)
# T4 = Turing (sm_75): bf16 is unsupported; fp16 is the only tensor-core dtype.
selected_dtype = 'fp16' if cap_major < 8 else 'bf16'
print('GPU            :', gpu_name)
print('compute cap    :', compute_capability)
print('VRAM (GB)      :', round(vram_gb, 1))
print('selected dtype :', selected_dtype)


In [ ]:
# --- Section 3: Budget gate - fail loudly BEFORE training (M2-P0-05, Q2) ---
MIN_VRAM_GB = 14.0
assert vram_gb >= MIN_VRAM_GB, (
    'Insufficient VRAM: %.1f GB < %.1f GB required for gpt-oss-20b QLoRA. '
    'Aborting before training rather than OOM-ing mid-run.' % (vram_gb, MIN_VRAM_GB)
)
assert PACKAGE['dtype'] == 'fp16', 'Package dtype must be fp16 (T4 is Turing/sm_75; bf16 unsupported).'
assert selected_dtype == 'fp16', 'This GPU is not Turing-class; the pinned recipe expects fp16 on a T4.'

max_seq_length = PACKAGE['sequence_length']
if vram_gb < 15.0 and max_seq_length > 512:
    print('VRAM %.1f GB < 15 GB - downgrading max_seq_length %d -> 512' % (vram_gb, max_seq_length))
    max_seq_length = 512
print('budget gate passed; max_seq_length =', max_seq_length)


In [ ]:
# --- Section 4: Install the pinned engine set via uv (M2-P0-06) ---
#
# INCIDENT REMEDIATION (GHARIBO-exp-001, first governed launch).
# The kernel reached KernelWorkerStatus.ERROR at THIS cell. The captured external log
# shows the failure was `CalledProcessError: uv pip install ... returned non-zero exit
# status 1` with the resolver's own reason discarded by `-qqq`.
#
# Root-cause class: DEPENDENCY_INSTALL_FAILURE_WITH_DIAGNOSTIC_SUPPRESSED. Two defects:
#   (a) `-qqq` suppressed the resolver output, so the real failure reason was lost and
#       the run was undiagnosable; and
#   (b) the naive resolver set re-resolved torch/triton against the default PyPI index,
#       discarding the Kaggle-provided +cu128 builds (local-version wheels are not on
#       that index).
# Both are corrected below, mirroring the install discipline already proven on Kaggle
# by the accepted M3C qualification harness (engine freeze unsloth-freeze-2026.09.15).
#
# Guarantees:
#   1. No -qqq on install commands - complete stdout + stderr are captured.
#   2. Every stage is dry-run with its EXACT arguments immediately before it executes.
#   3. On failure a bounded redacted diagnostic is persisted under /kaggle/working and
#      the raised error INCLUDES the real resolver/package reason (never "exit 1").
#   4. Preinstalled torch/triton are preserved and constraint-pinned, so no stage can
#      upgrade them.
#   5. triton_kernels is skipped on the Kaggle preserve path (upstream-aligned).
#
# The governed dependency SET is unchanged: every declared dependency is still installed
# at its declared spec. Only the resolution strategy is corrected.
import os, pathlib, re, shutil, subprocess, sys

WORKING = pathlib.Path('/kaggle/working')
if not WORKING.exists():
    WORKING = pathlib.Path('.')

INSTALL_DIAGNOSTIC_PATH = WORKING / 'install-diagnostic.json'

# Packages the Kaggle image already provides. Their wheels carry a local version tag
# (+cu128) that is NOT published on the default PyPI index, so re-resolving them
# silently replaces the CUDA build.
KAGGLE_PRESERVED_CANDIDATES = ('torch', 'triton')
# Source-built kernels the accepted qualification harness skips whenever the
# preinstalled torch/triton are preserved.
SKIP_WHEN_PRESERVED = ('triton_kernels',)


def redact(text):
    # Removes anything that looks like a credential before it is printed or stored.
    text = re.sub(r'hf_[A-Za-z0-9]{10,}', '<redacted-hf-token>', text)
    text = re.sub(r'(?i)(api[_-]?key|token|secret|password)(["\']?\s*[:=]\s*)([^\s"\',]+)',
                  r'\1\2<redacted>', text)
    return text


def scrub_paths(text):
    # The Kaggle input mount directory is the dataset slug; keep it out of the artifact.
    return text.replace('/kaggle/input/', '/kaggle/input/<dataset>/')


def run_install_command(cmd, phase, timeout=None):
    # Runs an install command with FULL observability.
    #
    # Unlike a bare subprocess.run(check=True), this captures the COMPLETE stdout and
    # stderr, redacts credentials, persists a bounded redacted diagnostic on failure,
    # and raises an exception that carries the actual resolver/package failure reason.
    display = scrub_paths(redact(' '.join(cmd)))
    print('$', display)
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if proc.returncode != 0:
        max_diag = 32000
        out = scrub_paths(redact(proc.stdout or ''))
        err = scrub_paths(redact(proc.stderr or ''))
        out_trim = out[-max_diag:]
        err_trim = err[-max_diag:]
        diagnostic = {
            'phase': phase,
            'command': display,
            'exit_code': proc.returncode,
            'stdout_redacted': out_trim,
            'stderr_redacted': err_trim,
        }
        try:
            import json as _json
            INSTALL_DIAGNOSTIC_PATH.write_text(_json.dumps(diagnostic, indent=1), encoding='utf-8')
        except Exception as exc:
            print('could not persist install diagnostic:', type(exc).__name__)
        raise RuntimeError(
            '%s failed (exit code %d).\n'
            '--- redacted stdout (last %d chars) ---\n%s\n'
            '--- redacted stderr (last %d chars) ---\n%s\n'
            'Diagnostic persisted to %s'
            % (phase, proc.returncode, len(out_trim), out_trim, len(err_trim), err_trim,
               INSTALL_DIAGNOSTIC_PATH.name)
        )
    return proc


print('Bootstrapping uv...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '-qqq', 'uv'], check=True)
UV = shutil.which('uv') or os.path.join(os.path.dirname(sys.executable), 'uv')
if not shutil.which('uv') and not os.path.exists(UV):
    raise RuntimeError('uv was installed but is not on PATH - cannot continue.')

# Kaggle has no active virtualenv, so uv needs an explicit target; if a venv IS active
# it must not be bypassed.
if os.environ.get('VIRTUAL_ENV'):
    TARGET_FLAGS = ['--python', sys.executable]
else:
    TARGET_FLAGS = ['--system', '--python', sys.executable]

# Turing-only build target: keeps any source build from emitting sm_80+ kernels a T4
# cannot load.
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
os.environ.setdefault('CMAKE_CUDA_ARCHITECTURES', '75')

ON_KAGGLE = os.path.isdir('/kaggle')


def module_present(modname):
    try:
        __import__(modname)
        return True
    except Exception:
        return False


PRESERVED = {}
if ON_KAGGLE:
    import importlib.metadata as _metadata
    for _name in KAGGLE_PRESERVED_CANDIDATES:
        if module_present(_name):
            PRESERVED[_name] = _metadata.version(_name)

SKIPPED = set()
if PRESERVED:
    SKIPPED.update(SKIP_WHEN_PRESERVED)


def spec_for(dep):
    # The exact install spec for a governed dependency (git specs are reassembled).
    spec = dep['spec']
    if dep['source'] == 'git':
        if spec.startswith('@git+') or spec.startswith('git+'):
            return spec.lstrip('@')
        if dep.get('url'):
            return 'git+' + dep['url'] + spec
        return spec.lstrip('@')
    return spec


install_specs = []
for _dep in PACKAGE['engine']['dependencies']:
    if _dep['name'] in PRESERVED:
        print('preserving preinstalled %s==%s (not re-resolved against PyPI)'
              % (_dep['name'], PRESERVED[_dep['name']]))
        continue
    if _dep['name'] in SKIPPED:
        print('skipping %s on the Kaggle preserve path (upstream-aligned)' % _dep['name'])
        continue
    install_specs.append(spec_for(_dep))

print('Installing pinned set:')
for _spec in install_specs:
    print('  ', _spec)

# Exact constraints stop any transitive dependency from upgrading the preserved builds.
CONSTRAINT_PATH = WORKING / 'preserved-constraints.txt'
CONSTRAINT_PATH.write_text(
    ''.join('%s==%s\n' % _item for _item in sorted(PRESERVED.items())), encoding='utf-8')
CONSTRAINT_FLAGS = ['--constraint', str(CONSTRAINT_PATH)] if PRESERVED else []

BASE = [UV, 'pip', 'install', *TARGET_FLAGS, '--no-cache-dir', *CONSTRAINT_FLAGS]

INSTALL_PLAN = [('install', [*BASE, *install_specs])]

# The dry-run probe validates the EXACT arguments of each stage before it runs. It is a
# resolver check, not a transitive-compatibility guarantee, but it turns an opaque late
# failure into an early, fully-reported one.
for _phase, _cmd in INSTALL_PLAN:
    run_install_command([*_cmd, '--dry-run'], _phase + '-dry-run')
    run_install_command(_cmd, _phase)

import importlib.metadata as _metadata_after
for _name, _version in PRESERVED.items():
    _actual = _metadata_after.version(_name)
    if _actual != _version:
        raise RuntimeError('preserved dependency changed: %s==%s -> %s'
                           % (_name, _version, _actual))

print('install complete')
print('preserved:', PRESERVED or 'none')
print('skipped:', sorted(SKIPPED) or 'none')


In [ ]:
# --- Section 5: Verify pinned versions + import smoke test (M2-P0-06, Q10) ---
#
# Incident remediation: the previous floor check compared versions as STRINGS, so a
# legitimate `torch 2.10.0` was reported as "< floor 2.8.0" (lexicographically "2.10.0"
# sorts below "2.8.0"), failing the gate on a correct environment. Floor comparison is
# now numeric over the PEP 440 release segments.
import importlib
from importlib.metadata import version as pkg_version, PackageNotFoundError


def release_tuple(value):
    # The comparable numeric release prefix ("2.10.0+cu128" -> (2, 10, 0)).
    head = value.split('+')[0].split('-')[0]
    parts = []
    for chunk in head.split('.'):
        digits = ''
        for ch in chunk:
            if ch.isdigit():
                digits += ch
            else:
                break
        if digits == '':
            break
        parts.append(int(digits))
    return tuple(parts)


def satisfies_floor(installed, floor):
    return release_tuple(installed) >= release_tuple(floor)


mismatches = []
for d in PACKAGE['engine']['dependencies']:
    if d['source'] != 'pip':
        continue
    name = d['name']
    try:
        installed = pkg_version(name)
    except PackageNotFoundError:
        mismatches.append('%s: not installed (spec %s)' % (name, d['spec']))
        continue
    resolved = d.get('resolved_version')
    if resolved:
        if installed != resolved:
            mismatches.append('%s: installed %s != pinned %s' % (name, installed, resolved))
    else:
        floor = d['spec'].split('>=')[1] if '>=' in d['spec'] else None
        if floor and not satisfies_floor(installed, floor):
            mismatches.append('%s: installed %s < floor %s' % (name, installed, floor))
        print('recorded %s==%s (spec %s)' % (name, installed, d['spec']))
assert not mismatches, 'Pinned dependency mismatch: ' + '; '.join(mismatches)
import torch, triton
print('import smoke test ok:', torch.__version__, triton.__version__)


In [ ]:
# --- Section 6: Dataset + split hash verification - hard-fail on mismatch (M2-P0-07) ---
#
# GOVERNED TEST POLICY. The package declares split_policy.test_held_out = true: the TEST
# payload is permanently held out and must never be uploaded, attached, opened, read,
# parsed, tokenized or used in any way. Its hash remains a metadata anchor
# (HASH_INTEGRITY_ONLY).
#
# Incident remediation. The previous version of this cell had two defects that made it
# impossible for the governed run to pass:
#   (a) it required train.jsonl AND validation.jsonl AND test.jsonl to be present and
#       recomputed the FULL dataset hash across all three - structurally impossible when
#       the TEST payload is deliberately absent; and
#   (b) it looked for the dataset only under /kaggle/working/dataset, while an attached
#       Kaggle Dataset is mounted under /kaggle/input/<slug>/.
# Both are corrected here.
import hashlib, pathlib

WORKING = pathlib.Path('/kaggle/working')
if not WORKING.exists():
    WORKING = pathlib.Path('.')


def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


SPLIT_POLICY = PACKAGE['dataset'].get('split_policy') or {}
TEST_HELD_OUT = bool(SPLIT_POLICY.get('test_held_out'))
EXPECTED_SPLIT_HASHES = PACKAGE['dataset']['split_hashes']


def locate_dataset_dir():
    # Finds the directory holding the governed split files. An attached Kaggle Dataset
    # is mounted under /kaggle/input/<slug>/, so the input root is searched recursively;
    # a locally staged ./dataset layout is also accepted.
    candidates = [WORKING / 'dataset', pathlib.Path('dataset')]
    input_root = pathlib.Path('/kaggle/input')
    if input_root.is_dir():
        candidates.append(input_root)
        candidates.extend(sorted(p for p in input_root.rglob('*') if p.is_dir()))
    for candidate in candidates:
        if (candidate / 'train.jsonl').is_file():
            return candidate
    return None


DATA_DIR = locate_dataset_dir()
assert DATA_DIR is not None, (
    'the governed dataset was not found. Attach the private Kaggle Dataset carrying '
    'train.jsonl and validation.jsonl (TEST must NOT be present), or stage ./dataset.'
)

if TEST_HELD_OUT:
    split_names = ('train', 'validation')
    # A TEST payload anywhere in the execution bundle is a policy violation: stop.
    assert not (DATA_DIR / 'test.jsonl').exists(), (
        'TEST POLICY VIOLATION: test.jsonl is present in the execution bundle. TEST is '
        'permanently held out (HASH_INTEGRITY_ONLY) and must never be uploaded, attached '
        'or read. Remove it, rebuild the bundle and re-validate before launching.'
    )
else:
    split_names = ('train', 'validation', 'test')

split_lines = {}
for name in split_names:
    path = DATA_DIR / (name + '.jsonl')
    assert path.exists(), 'required split file missing: %s' % path
    with open(path, 'r', encoding='utf-8') as fh:
        split_lines[name] = [ln for ln in fh.read().split('\n') if ln.strip() != '']

all_hashes = []
for name in split_names:
    line_hashes = sorted(sha256_text(ln) for ln in split_lines[name])
    actual = sha256_text('\n'.join(line_hashes))
    expected = EXPECTED_SPLIT_HASHES[name]
    assert actual == expected, 'Split hash mismatch for %s: %s != %s' % (name, actual, expected)
    all_hashes.extend(line_hashes)

TRAIN_VALIDATION_HASH = sha256_text('\n'.join(sorted(all_hashes)))

if TEST_HELD_OUT:
    # The full dataset hash spans all three splits, so it CANNOT be recomputed without the
    # held-out TEST payload - and recomputing it would require reading TEST, which the
    # policy forbids. The TRAIN+VALIDATION content identity is recorded under its own hash
    # and the governed full dataset hash stays a metadata anchor only.
    print('dataset + split hashes verified (TRAIN + VALIDATION only)')
    print('  train=%d validation=%d test=ABSENT (permanently held out)'
          % (len(split_lines['train']), len(split_lines['validation'])))
    print('  train+validation content hash: %s' % TRAIN_VALIDATION_HASH)
    print('  dataset hash (metadata anchor, TEST never read): %s'
          % PACKAGE['dataset']['dataset_hash'])
    print('  TEST usage: HASH_INTEGRITY_ONLY')
else:
    dataset_hash = TRAIN_VALIDATION_HASH
    assert dataset_hash == PACKAGE['dataset']['dataset_hash'], (
        'Dataset hash mismatch: %s != %s' % (dataset_hash, PACKAGE['dataset']['dataset_hash'])
    )
    print('dataset + split hashes verified')
    print('  train=%d validation=%d test=%d' % (
        len(split_lines['train']), len(split_lines['validation']), len(split_lines['test'])))


In [ ]:
# --- Section 7: Dataset record -> Harmony text ---
# The package explicitly declares the physical JSONL representation.
# harmony-messages-v1 is the governed Gold representation proven during
# the accepted real-Kaggle qualification.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    PACKAGE['loader_model_id']
)

reasoning_effort = PACKAGE['harmony']['reasoning_effort']
hidden_channels = set(
    PACKAGE['harmony']['hidden_channels']
)

assert 'analysis' in hidden_channels, (
    'Package must declare the analysis channel as hidden'
)

record_format = PACKAGE['dataset'].get('record_format')
if record_format is None and PACKAGE['schema_version'] == '1.0.0':
    record_format = 'canonical-record-v1'

assert record_format in (
    'canonical-record-v1',
    'harmony-messages-v1',
), 'Unsupported dataset.record_format: %r' % (record_format,)


def canonical_record_to_messages(record):
    messages = [
        {
            'role': 'developer',
            'content': (
                'Produce a well-structured, accurate answer '
                'grounded in the provided input and context.'
            ),
        }
    ]

    user = record.get('input') or ''

    if record.get('context'):
        user += '\n\nContext:\n' + record['context']

    messages.append({
        'role': 'user',
        'content': user,
    })

    if record.get('reasoning'):
        messages.append({
            'role': 'assistant',
            'channel': 'analysis',
            'content': record['reasoning'],
        })

    final = (
        record.get('chosen_output')
        or record.get('expected_output')
    )

    if final is not None:
        messages.append({
            'role': 'assistant',
            'channel': 'final',
            'content': final,
        })

    return messages


def governed_harmony_messages(record):
    raw = record.get('messages')

    assert isinstance(raw, list) and raw, (
        'harmony-messages-v1 requires a non-empty messages[]'
    )

    messages = []

    for index, message in enumerate(raw):
        assert isinstance(message, dict), (
            'messages[%d] must be an object' % index
        )

        role = message.get('role')
        content = message.get('content')

        assert isinstance(role, str) and role, (
            'messages[%d].role must be a non-empty string'
            % index
        )

        assert isinstance(content, str), (
            'messages[%d].content must be a string'
            % index
        )

        # Same role/content representation proven by qualification v6.
        messages.append({
            'role': role,
            'content': content,
        })

    return messages


def render_texts(records):
    texts = []

    for index, record in enumerate(records):

        if record_format == 'harmony-messages-v1':
            messages = governed_harmony_messages(record)

            rendered = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )

        else:
            messages = canonical_record_to_messages(record)

            rendered = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                reasoning_effort=reasoning_effort,
            )

        assert isinstance(rendered, str) and rendered, (
            'Tokenizer produced empty Harmony text '
            'for record %d' % index
        )

        texts.append(rendered)

        # Never print record content or hidden reasoning.
        if index % 25 == 0:
            print('rendered record index', index)

    return texts


train_records = [
    json.loads(line)
    for line in split_lines['train']
]

train_texts = render_texts(train_records)

print(
    'rendered',
    len(train_texts),
    'training texts from',
    record_format,
    '(record content not printed)',
)


In [ ]:
# --- Section 8: Load the gpt-oss 4-bit representation (M2-P0-08) ---
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=PACKAGE['loader_model_id'],
    dtype=None,            # auto-detect -> resolves to fp16 on a T4
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    full_finetuning=False,
)
print('model loaded:', PACKAGE['loader_model_id'])


In [ ]:
# --- Section 9: QLoRA adapters - r/alpha/target_modules FROM the package (M2-P0-09) ---
lora = PACKAGE['lora']
model = FastLanguageModel.get_peft_model(
    model,
    r=lora['r'],
    target_modules=list(lora['target_modules']),
    lora_alpha=lora['alpha'],
    lora_dropout=lora['dropout'],
    bias=lora['bias'],
    use_gradient_checkpointing='unsloth',
    random_state=PACKAGE['seed'],
    use_rslora=False,
    loftq_config=None,
)
print('LoRA r=%d alpha=%d modules=%s' % (lora['r'], lora['alpha'], lora['target_modules']))


In [ ]:
# --- Section 10: SFT config from the package (M2-P0-09, M2-P0-10) ---
from trl import SFTConfig, SFTTrainer
from datasets import Dataset as HFDataset

cp = PACKAGE['checkpoint_policy']
sft_kwargs = dict(
    per_device_train_batch_size=PACKAGE['batch']['per_device_train_batch_size'],
    gradient_accumulation_steps=PACKAGE['batch']['gradient_accumulation_steps'],
    warmup_steps=PACKAGE['warmup_steps'],
    learning_rate=PACKAGE['learning_rate'],
    logging_steps=1,
    optim=PACKAGE['optimizer'],
    weight_decay=PACKAGE['weight_decay'],
    lr_scheduler_type=PACKAGE['lr_scheduler_type'],
    seed=PACKAGE['seed'],
    output_dir=str(WORKING / 'outputs'),
    report_to='none',
    save_strategy=cp['save_strategy'],
    save_steps=cp['save_steps'],
    save_total_limit=cp['save_total_limit'],
    fp16=(PACKAGE['dtype'] == 'fp16'),
    bf16=False,
)
if PACKAGE['epochs'] is not None:
    sft_kwargs['num_train_epochs'] = PACKAGE['epochs']
if PACKAGE['max_steps'] is not None:
    sft_kwargs['max_steps'] = PACKAGE['max_steps']

train_dataset = HFDataset.from_dict({'text': train_texts})
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_dataset, args=SFTConfig(**sft_kwargs))
print('SFT config ready (save_strategy=%s, save_steps=%s, save_total_limit=%s)' % (
    cp['save_strategy'], cp['save_steps'], cp['save_total_limit']))


In [ ]:
# --- Section 11: Resume from checkpoint when supplied by the package (M2-P0-11) ---
resume_from_checkpoint = PACKAGE['checkpoint_policy']['resume_from_checkpoint']
if resume_from_checkpoint:
    print('RESUMING from checkpoint:', resume_from_checkpoint)
else:
    print('fresh run - no resume point supplied')


In [ ]:
# --- Section 12: Train (M2-P0-10) ---
train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
print('training finished')


In [ ]:
# --- Section 13: Save adapter + trainer state + metrics + manifest (M2-P0-10, M2-P0-20) ---
ADAPTER_DIR = WORKING / 'adapter'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
trainer.save_model(str(WORKING / 'outputs' / 'final'))

metrics = getattr(trainer.state, 'log_history', [])
with open(WORKING / 'metrics.json', 'w', encoding='utf-8') as fh:
    json.dump(metrics, fh, indent=2, sort_keys=True)

manifest = dict(PACKAGE)
manifest['environment_metadata'] = {
    'os': os.name,
    'python_version': sys.version.split()[0],
    'packages': {'torch': torch.__version__, 'triton': triton.__version__},
    'gpu': gpu_name,
    'cuda': torch.version.cuda,
}
manifest['resume_from_checkpoint'] = resume_from_checkpoint
with open(WORKING / 'manifest.json', 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
print('artifacts saved under', WORKING)


In [ ]:
# --- Section 14: CHECKSUMS.sha256 - per-file + rollup (Q9) ---
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

entries = []
for path in sorted(WORKING.rglob('*')):
    if path.is_file() and path.name != 'CHECKSUMS.sha256':
        rel = path.relative_to(WORKING).as_posix()
        entries.append((rel, sha256_file(path)))
rollup = sha256_text('\n'.join(sorted('%s\t%s' % (rel, digest) for rel, digest in entries)))
lines_out = ['%s  %s' % (digest, rel) for rel, digest in entries]
lines_out.append('# rollup  ' + rollup)
with open(WORKING / 'CHECKSUMS.sha256', 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(lines_out) + '\n')
print('CHECKSUMS.sha256 written; rollup =', rollup)


In [ ]:
# --- Section 15: Optional PRIVATE HF upload via Kaggle Secrets (M2-P0-12) ---
# The token is read by NAME and never printed, never written to any file.
dest = PACKAGE.get('artifact_destination')
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    secret_name = (dest or {}).get('token_secret_name') or 'HF_TOKEN'
    hf_token = UserSecretsClient().get_secret(secret_name)
except Exception:
    hf_token = None

if dest and dest.get('kind') == 'hf' and dest.get('private') is True and hf_token:
    model.push_to_hub_merged(dest['repo_id'], tokenizer=tokenizer, token=hf_token, save_method='mxfp4')
    print('uploaded adapter to private HF repo:', dest['repo_id'])
else:
    print('no HF destination configured (or secret absent) - local /kaggle/working export is the result')
del hf_token


In [ ]:
# --- Section 16: Finalize - completion marker + logs under /kaggle/working (M2-P0-19, M2-P0-20) ---
env_meta = manifest['environment_metadata']
with open(WORKING / 'training.log', 'a', encoding='utf-8') as fh:
    fh.write('experiment_id=' + PACKAGE['experiment_id'] + '\n')
    fh.write('package_id=' + PACKAGE['package_id'] + '\n')
    fh.write('gpu=' + str(env_meta['gpu']) + '\n')
    fh.write('cuda=' + str(env_meta['cuda']) + '\n')
    fh.write('status=COMPLETED\n')
with open(WORKING / 'COMPLETED', 'w', encoding='utf-8') as fh:
    fh.write(PACKAGE['package_id'] + '\n')
print('run complete - outputs persisted under /kaggle/working')
print('NOTE: run as a committed / Save-Version notebook so /kaggle/working persists.')
